Jupyter Notebook: DINOv2 and MobileNet Model Explorer
Explores model parameters, layers, and outputs for different input sizes

In [5]:
import torch
import torch.nn as nn
import torchvision.models as models
from transformers import AutoModel, AutoConfig
import numpy as np
from collections import OrderedDict
import time

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

PyTorch version: 2.7.1+cu118
CUDA available: True


In [2]:
def count_parameters(model):
    """Count total and trainable parameters"""
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total_params, trainable_params

def print_model_summary(model, model_name, input_size):
    """Print comprehensive model summary"""
    total_params, trainable_params = count_parameters(model)
    
    print(f"\n{'='*80}")
    print(f"Model: {model_name}")
    print(f"Input Size: {input_size}x{input_size}")
    print(f"{'='*80}")
    print(f"Total Parameters: {total_params:,}")
    print(f"Trainable Parameters: {trainable_params:,}")
    print(f"Model Size (MB): {total_params * 4 / (1024**2):.2f}")  # 4 bytes per parameter
    print(f"\nNumber of Layers: {len(list(model.modules()))}")
    
    return total_params, trainable_params

def get_layer_names(model, max_layers=np.inf):
    """Get first N layer names and their shapes"""
    layer_names = []
    for name, module in model.named_modules():
        if name and len(layer_names) < max_layers:
            layer_names.append(f"{name}: {module.__class__.__name__}")
    return layer_names

def test_model_output(model, input_size, model_name):
    """Test model with fake input and print output shape"""
    # Create fake input (batch_size=1, channels=3, height, width)
    fake_input = torch.randn(1, 3, input_size, input_size)
    
    print(f"\nTesting {model_name} with input shape: {fake_input.shape}")
    
    model.eval()
    with torch.no_grad():
        try:
            output = model(fake_input)
            if isinstance(output, dict):
                print("Output is a dictionary:")
                for key, value in output.items():
                    if isinstance(value, torch.Tensor):
                        print(f"  {key}: {value.shape}")
            elif isinstance(output, torch.Tensor):
                print(f"Output shape: {output.shape}")
            else:
                print(f"Output type: {type(output)}")
            return output
        except Exception as e:
            print(f"Error during forward pass: {e}")
            return None


In [3]:
my_models = {
    # MobileNet variants
    'MobileNetV2': models.mobilenet_v2(pretrained=True),
    'MobileNetV3-Small': models.mobilenet_v3_small(pretrained=True),
    'MobileNetV3-Large': models.mobilenet_v3_large(pretrained=True),
    
    # EfficientNet variants (available in torchvision)
    'EfficientNet-B0': models.efficientnet_b0(pretrained=True),
    'EfficientNet-B1': models.efficientnet_b1(pretrained=True),
    'EfficientNet-B2': models.efficientnet_b2(pretrained=True),
    'EfficientNet-B3': models.efficientnet_b3(pretrained=True),
    'EfficientNet-B4': models.efficientnet_b4(pretrained=True),
    
    # ShuffleNet variants
    'ShuffleNet-V2-x0.5': models.shufflenet_v2_x0_5(pretrained=True),
    'ShuffleNet-V2-x1.0': models.shufflenet_v2_x1_0(pretrained=True),
    'ShuffleNet-V2-x1.5': models.shufflenet_v2_x1_5(pretrained=True),
    'ShuffleNet-V2-x2.0': models.shufflenet_v2_x2_0(pretrained=True),
    
    # RegNet variants
    'RegNet-Y-400MF': models.regnet_y_400mf(pretrained=True),
    'RegNet-Y-800MF': models.regnet_y_800mf(pretrained=True),
    'RegNet-Y-1.6GF': models.regnet_y_1_6gf(pretrained=True),
    'RegNet-Y-3.2GF': models.regnet_y_3_2gf(pretrained=True),
    
    # # SqueezeNet
    # 'SqueezeNet-1.0': models.squeezenet1_0(pretrained=False),
    # 'SqueezeNet-1.1': models.squeezenet1_1(pretrained=False),
    
    # # MNASNet
    # 'MNASNet-0.5': models.mnasnet0_5(pretrained=False),
    # 'MNASNet-1.0': models.mnasnet1_0(pretrained=False),
    
    # DINOv2 models (available via torch hub)
    'dinov2_vits14': torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14'),
    'dinov2_vitb14': torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14'),

    # # DINOv3 models
    # 'dinov3_vits16': AutoModel.from_pretrained('facebook/dinov3-small'),
    # 'dinov3_vitb16': AutoModel.from_pretrained('facebook/dinov3-base'),
}

c:\Users\szige\Google Drive\02_study\0_BME_alkmat\semester_1\ai_in_ds\kaggle_csiro\myenv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\szige\Google Drive\02_study\0_BME_alkmat\semester_1\ai_in_ds\kaggle_csiro\myenv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
c:\Users\szige\Google Drive\02_study\0_BME_alkmat\semester_1\ai_in_ds\kaggle_csiro\myenv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0

In [30]:
input_sizes = [224, 518]

input_size = input_sizes[1]

for model_name, model in my_models.items():
    print(f"\n{'='*80}")
    print(f"Analyzing: {model_name} with {input_size}x{input_size} input")
    print(f"{'='*80}")

    # Print model summary
    total_params, trainable_params = print_model_summary(model, model_name, input_size)

    # Print first and last few layers
    # print("len(model.layers):", len(model.layers))
    layer_names = get_layer_names(model)
    print(f"\nLayers:")
    for i, layer in enumerate(layer_names):
        if (i > 1) and (i < len(layer_names) - 10):  # Limit to first 2 and last 10 layers for brevity
            continue
        if i == 2:
            print("  ...")
        print(f"  {i}. {layer}")

    # Test with fake input
    t0 = time.perf_counter()
    output = test_model_output(model, input_size, model_name)
    t1 = time.perf_counter()
    print(f"\nForward pass time: {t1 - t0:.4f} seconds, shape: {output.shape if output is not None and isinstance(output, torch.Tensor) else 'N/A'}")

    # mobilenet_results[model_name][input_size] = {
    #     'total_params': total_params,
    #     'trainable_params': trainable_params,
    #     'output_shape': output.shape if output is not None and isinstance(output, torch.Tensor) else None
    # }


Analyzing: MobileNetV2 with 518x518 input

Model: MobileNetV2
Input Size: 518x518
Total Parameters: 3,504,872
Trainable Parameters: 3,504,872
Model Size (MB): 13.37

Number of Layers: 213

Layers:
  0. features: Sequential
  1. features.0: Conv2dNormActivation
  202. features.17.conv.1.2: ReLU6
  203. features.17.conv.2: Conv2d
  204. features.17.conv.3: BatchNorm2d
  205. features.18: Conv2dNormActivation
  206. features.18.0: Conv2d
  207. features.18.1: BatchNorm2d
  208. features.18.2: ReLU6
  209. classifier: Sequential
  210. classifier.0: Dropout
  211. classifier.1: Linear

Testing MobileNetV2 with input shape: torch.Size([1, 3, 518, 518])
Output shape: torch.Size([1, 1000])

Forward pass time: 0.4029 seconds, shape: torch.Size([1, 1000])

Analyzing: MobileNetV3-Small with 518x518 input

Model: MobileNetV3-Small
Input Size: 518x518
Total Parameters: 2,542,856
Trainable Parameters: 2,542,856
Model Size (MB): 9.70

Number of Layers: 209

Layers:
  0. features: Sequential
  1. fe

In [31]:
my_models['dinov2_vits14'].embed_dim

384

In [ ]:
from huggingface_hub import login
login("") #"YOUR_HF_TOKEN_HERE")

HTTPError: Invalid user token.

In [ ]:
# SELECT YOUR SIZE HERE:
# model_name = "facebook/dinov3-vitb16-pretrain-lvd1689m"  # Base
model_name = "facebook/dinov3-vitl16-pretrain-lvd1689m"  # Large
# model_name = "facebook/dinov3-vith16plus-pretrain-lvd1689m" # Giant (Requires ~24GB VRAM)

try:
    print(f"Downloading {model_name}...")
    
    # Load Config first to verify access
    config = AutoConfig.from_pretrained(model_name)
    
    # Load Model
    fe_model = AutoModel.from_pretrained(model_name)
    
    print("✅ Success! Model loaded.")
    
except OSError as e:
    print("❌ Error: Still getting 401/404?")
    print("1. Check your spelling exactly.")
    print("2. Try logging in: run `huggingface-cli login` in terminal with your User Access Token.")
    print(f"Details: {e}")

❌ Error: Still getting 401/404?
1. Check your spelling exactly.
2. Try logging in: run `huggingface-cli login` in terminal with your User Access Token.
Details: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/facebook/dinov3-vitl16-pretrain-lvd1689m.
401 Client Error. (Request ID: Root=1-6977e579-710ef3f32d8b89eb362675cb;12fd6b43-b5ca-4a0f-ac7d-37754f6eb106)

Cannot access gated repo for url https://huggingface.co/facebook/dinov3-vitl16-pretrain-lvd1689m/resolve/main/config.json.
Access to model facebook/dinov3-vitl16-pretrain-lvd1689m is restricted. You must have access to it and be authenticated to access it. Please log in.


In [8]:
model_dinov2_vitl14 = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitl14')
model_dinov2_vitl14.embed_dim

Using cache found in C:\Users\szige/.cache\torch\hub\facebookresearch_dinov2_main
C:\Users\szige/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
C:\Users\szige/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
C:\Users\szige/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


1024

Using cache found in C:\Users\szige/.cache\torch\hub\facebookresearch_dinov2_main


RuntimeError: Cannot find callable dinov2_vith14 in hubconf